# Temporal và graph có làm normal snippet giống anomaly hơn?
Notebook này kiểm tra checkpoint VadCLIP đã train, không huấn luyện lại. Nhãn frame của test set chỉ dùng để phân tích và can thiệp kiểm chứng.

`false-high` là normal snippet nằm trong top 15% score của một abnormal video; `true-high` là anomaly snippet trong top 15%; `true-low` là normal snippet ngoài top 15%. Kết quả lưu trong `code/results/analyze/temporal_graph_leakage/`.

## 1. Kết nối Drive và chọn checkpoint
Mặc định dùng baseline tự train. Nếu muốn phân tích mô hình khác cùng kiến trúc VadCLIP gốc, chỉ đổi `MODEL_PATH` và `OUTPUT_DIR`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
MODEL_PATH = SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
TEST_LIST = LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'
GT_PATH = LIST_DIR / 'gt_ucf.npy'
OUTPUT_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'temporal_graph_leakage'
TOP_K_RATIO = 0.15
MAX_ABNORMAL_VIDEOS = 0  # 0 = toàn bộ abnormal test videos; 5 = chạy thử nhanh

print('Project:', PROJECT_ROOT)
print('Model:', MODEL_PATH)
print('Feature root:', FEATURE_ROOT)
print('Output:', OUTPUT_DIR)


## 2. Chuẩn bị thư viện và kiểm tra đầu vào
Notebook dùng feature đã giải nén trong `/content/UCFClipFeatures`. Nếu còn thiếu file và có `UCFClipFeatures.zip` trên Drive, `unzip -n` chỉ giải nén những file chưa có.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'scipy', 'matplotlib', 'pandas'], check=True)
import torch
import csv
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU trước khi chạy.')
print('GPU:', torch.cuda.get_device_name(0))

ANALYZER = SRC_DIR / 'ucf_analyze_temporal_graph_leakage.py'
for path in (MODEL_PATH, TEST_LIST, GT_PATH, ANALYZER):
    if not path.exists():
        raise FileNotFoundError(f'Thiếu đầu vào: {path}')
with TEST_LIST.open(newline='', encoding='utf-8-sig') as handle:
    test_rows = list(csv.DictReader(handle))
missing_features = [row['path'] for row in test_rows if not (FEATURE_ROOT / row['path']).is_file()]
if missing_features:
    archive = PROJECT_ROOT / 'UCFClipFeatures.zip'
    if not archive.is_file():
        raise FileNotFoundError(f'Thiếu {len(missing_features)} test features; không thấy {archive}. Ví dụ: {missing_features[:5]}')
    print(f'Còn thiếu {len(missing_features)} test features; tiếp tục giải nén từ {archive}')
    subprocess.run(['unzip', '-q', '-n', str(archive), '-d', '/content'], check=True)
missing_features = [row['path'] for row in test_rows if not (FEATURE_ROOT / row['path']).is_file()]
if missing_features:
    raise FileNotFoundError(f'Archive chưa cung cấp đủ {len(missing_features)} features. Ví dụ: {missing_features[:5]}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Đầu vào đã sẵn sàng.')


## 3. Chạy phân tích
Mỗi video được chạy qua VadCLIP một lần. Script ghi feature ở ba tầng, attention đến các anomaly snippet, trọng số của hai graph, rồi tính lại graph output sau khi chặn cạnh từ normal query đến anomaly source. Một lần chặn ngẫu nhiên cùng số lượng normal source được dùng làm đối chứng.

Các phép chặn chỉ thực hiện khi phân tích và không làm thay đổi checkpoint.

In [ ]:
command = [
    sys.executable, '-u', str(ANALYZER),
    '--model-path', str(MODEL_PATH),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(GT_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--top-k-ratio', str(TOP_K_RATIO),
    '--max-abnormal-videos', str(MAX_ABNORMAL_VIDEOS),
]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, cwd=SRC_DIR, check=True)


## 4. Xem kết quả
`delta` là cosine(false-high, true-high) trừ cosine(false-high, true-low). `mass` là phần trọng số đi từ normal query đến anomaly source. `excess` đã trừ tỷ lệ anomaly source sẵn có trong vùng attention hoặc graph. Số âm trong can thiệp nghĩa là score hoặc độ gần anomaly giảm sau khi bỏ cạnh.

So sánh đầu vào CLIP với feature cuối là một chỉ báo về thay đổi quan hệ tương đối; attention mass và graph intervention giúp kiểm tra cơ chế, không phải bằng chứng nhân quả hoàn chỉnh cho mọi loại lan truyền.

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
print('Số abnormal video:', summary['abnormal_videos_analyzed'])
print('Số video có đủ ba nhóm:', summary['videos_with_all_three_groups'])
print('Sai số replay graph tối đa:', summary['max_graph_replay_error'])
print('Tỷ lệ video có delta tăng từ CLIP đến cuối:', summary['raw_to_final_positive_change_fraction'])
print('Thay đổi delta CLIP -> final (trung bình, CI95):', summary['raw_to_final_delta_change'])

stage = pd.read_csv(OUTPUT_DIR / 'per_video_stage_similarity.csv')
display(stage.groupby('stage', sort=False)[['false_high_to_true_high', 'false_high_to_true_low', 'delta']].agg(['mean', 'std', 'count']).round(4))

exposure = pd.read_csv(OUTPUT_DIR / 'per_video_exposure.csv')
display(exposure.groupby('group')[['temporal_anomaly_mass', 'temporal_excess', 'similarity_graph_anomaly_mass', 'similarity_graph_excess', 'distance_graph_anomaly_mass', 'distance_graph_excess']].mean().round(4))
print('So sánh theo cặp trên cùng video, false-high trừ true-low:')
display(pd.DataFrame(summary['paired_exposure_false_high_minus_true_low']).T.round(4))

intervention = pd.read_csv(OUTPUT_DIR / 'per_video_intervention.csv')
display(intervention.groupby('group')[['remove_anomaly_score_change', 'remove_normal_score_change', 'remove_anomaly_true_high_affinity_change']].mean().round(4))
print('Hiệu ứng can thiệp trên cùng video, false-high trừ true-low:')
display(pd.DataFrame(summary['paired_intervention_false_high_minus_true_low']).T.round(4))
print('Tương quan exposure và thay đổi độ gần anomaly:')
display(pd.Series(summary['exposure_affinity_correlations'], name='Pearson r').to_frame().round(4))

for filename in ('01_feature_affinity_by_stage.png', '02_anomaly_source_exposure.png', '03_graph_intervention.png', '04_graph_exposure_vs_drift.png'):
    image_path = OUTPUT_DIR / filename
    if image_path.exists():
        display(Image(filename=str(image_path)))
print('Các file CSV, JSON và ảnh nằm tại:', OUTPUT_DIR)


## 5. So sánh mean toàn video và mean chỉ từ normal snippet
Cell dưới đây chạy độc lập với các cell phân tích model phía trên: không cần GPU hoặc checkpoint. Nó tính hai mean trên feature CLIP đã trích xuất, rồi dùng nhãn test để kiểm tra khả năng xếp hạng anomaly trong từng video. Mean chỉ từ normal snippet là **oracle diagnostic**, không dùng được trực tiếp khi inference và không chứng minh thành phần chung là background.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Finetune VadCLIP')
feature_root = Path('/content/UCFClipFeatures')
test_list = root / 'VadCLIP/list/ucf_CLIP_rgbtest_relative.csv'
gt_path = root / 'VadCLIP/list/gt_ucf.npy'
analyzer = root / 'VadCLIP/src/ucf_analyze_video_centering.py'
output_dir = root / 'code/results/analyze/video_centering'
for path in (test_list, gt_path, analyzer):
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu file: {path}')

with test_list.open(newline='', encoding='utf-8-sig') as handle:
    test_rows = list(csv.DictReader(handle))
missing = [row['path'] for row in test_rows if not (feature_root / row['path']).is_file()]
if missing:
    archive = root / 'UCFClipFeatures.zip'
    if not archive.is_file():
        raise FileNotFoundError(f'Thiếu {len(missing)} feature và không thấy ZIP: {archive}')
    print(f'Thiếu {len(missing)} feature; giải nén tiếp từ {archive}', flush=True)
    subprocess.run(['unzip', '-q', '-n', str(archive), '-d', '/content'], check=True)
missing = [row['path'] for row in test_rows if not (feature_root / row['path']).is_file()]
if missing:
    raise FileNotFoundError(f'Vẫn thiếu {len(missing)} feature sau giải nén. Ví dụ: {missing[:5]}')

subprocess.run([
    sys.executable, '-u', str(analyzer),
    '--feature-root', str(feature_root),
    '--test-list', str(test_list),
    '--gt-path', str(gt_path),
    '--output-dir', str(output_dir),
], check=True)
summary = json.loads((output_dir / 'summary.json').read_text(encoding='utf-8'))
print('Mean toàn bộ, AUC trung bình theo video:', summary['macro_video_auc']['all_cosine']['mean'])
print('Mean chỉ normal (oracle), AUC trung bình theo video:', summary['macro_video_auc']['normal_cosine']['mean'])
print('Độ giống giữa hai mean trên video abnormal:', summary['mean_all_vs_normal_cosine_abnormal_videos']['mean'])
print('Kết quả đã lưu tại:', output_dir)


## 6. Kiểm tra model có dùng thứ tự thời gian không
Cell này giữ nguyên feature của 20 cặp false-high/true-high, xáo trộn thứ tự trong từng cửa sổ 256 snippet, rồi so score trước và sau. Chạy cell thiết lập ở mục 1 trước; không cần chạy lại các cell phân tích nặng ở trên.


In [ ]:
ORDER_ANALYZER = SRC_DIR / 'ucf_analyze_temporal_order.py'
ORDER_OUTPUT_DIR = OUTPUT_DIR / 'order_shuffle_20_videos'
for path in (MODEL_PATH, TEST_LIST, GT_PATH, ORDER_ANALYZER):
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu file: {path}')
order_command = [
    sys.executable, '-u', str(ORDER_ANALYZER),
    '--model-path', str(MODEL_PATH),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(GT_PATH),
    '--output-dir', str(ORDER_OUTPUT_DIR),
    '--shuffle-repeats', '5',
]
print('Running:', ' '.join(order_command), flush=True)
subprocess.run(order_command, cwd=SRC_DIR, check=True)


## 7. Tách ảnh hưởng của temporal và graph
Cell này dùng cùng checkpoint và 20 video đã chọn. Nó so score classifier giữa thứ tự gốc và shuffle, đồng thời thử similarity graph riêng, distance graph riêng, adjacency chỉ nối mỗi snippet với chính nó, và shuffle khi giữ nguyên graph links theo danh tính snippet. Không train lại và không tính AUC. Lưu ý: adjacency chỉ nối chính nó vẫn giữ convolution residual trong các graph layer; vì vậy đây là phép kiểm soát graph message passing, không phải xóa toàn bộ xử lý cục bộ.

In [ ]:
FACTORIAL_ANALYZER = SRC_DIR / 'ucf_analyze_temporal_graph_factorial.py'
FACTORIAL_OUTPUT_DIR = OUTPUT_DIR / 'temporal_graph_factorial_20_videos'
if not FACTORIAL_ANALYZER.is_file():
    raise FileNotFoundError(f'Thiếu analyzer: {FACTORIAL_ANALYZER}')
factorial_command = [
    sys.executable, '-u', str(FACTORIAL_ANALYZER),
    '--model-path', str(MODEL_PATH),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(GT_PATH),
    '--output-dir', str(FACTORIAL_OUTPUT_DIR),
    '--shuffle-repeats', '5',
]
print('Running:', ' '.join(factorial_command), flush=True)
subprocess.run(factorial_command, cwd=SRC_DIR, check=True)


## 8. Phân tích thống nhất đóng góp của từng tầng
Chạy trên toàn bộ test set. So sánh feature ở đầu vào CLIP, sau temporal attention, sau graph và tại classifier; đồng thời can thiệp temporal/graph để đo thay đổi của logit và score. Các snippet top 15% được chọn một lần từ model đầy đủ rồi giữ cố định giữa các điều kiện. Ground truth chỉ dùng để phân nhóm và đánh giá sau inference.

In [ ]:
STAGE_ANALYZER = SRC_DIR / 'ucf_analyze_stage_contributions.py'
STAGE_OUTPUT_DIR = OUTPUT_DIR / 'stage_contributions_all_test'
STAGE_DEPENDENCIES = [
    STAGE_ANALYZER,
    SRC_DIR / 'ucf_analyze_targeted_temporal_context.py',
    SRC_DIR / 'ucf_analyze_temporal_graph_factorial.py',
    SRC_DIR / 'ucf_analyze_temporal_order.py',
]
for path in STAGE_DEPENDENCIES:
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu file nguồn cần upload: {path}')
stage_command = [
    sys.executable, '-u', str(STAGE_ANALYZER),
    '--model-path', str(MODEL_PATH),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(GT_PATH),
    '--output-dir', str(STAGE_OUTPUT_DIR),
    '--top-k-ratio', '0.15',
]
print('Running:', ' '.join(stage_command), flush=True)
subprocess.run(stage_command, cwd=SRC_DIR, check=True)
